# 02 - Xử lý bảng ORDER (chuẩn 3NF, tiêu chuẩn Silver)

**Nguồn dữ liệu:** `orders_enriched_silver.csv`
**Bảng đích:** `ORDER` theo lược đồ 3NF (`Luoc_do_quan_he_3NF.docx`, mục 7)
**Bảng phụ thuộc:** `CUSTOMER.csv` (đã xuất ở notebook 01) — dùng để kiểm tra ràng buộc khóa ngoại `customer_id`.

| Thuộc tính đích | Nguồn | Ghi chú |
|---|---|---|
| order_id (PK) | order_id | giữ nguyên |
| order_date | order_date | parse về kiểu date |
| customer_id (FK -> CUSTOMER) | customer_id | giữ nguyên |
| zip (FK -> GEOGRAPHY) | zip | giữ nguyên |
| order_status | order_status | chuẩn hóa chuỗi |
| device_type | device_type | chuẩn hóa chuỗi |
| order_source | order_source | chuẩn hóa chuỗi |
| sales_employee_id (FK -> SALES_EMPLOYEE) | sales_employee_id | chuẩn hóa chuỗi |

**Cột bị loại bỏ khỏi bảng ORDER** (không thuộc lược đồ 3NF của bảng này):
- `city`, `region`, `district`: dư thừa vì suy ra được qua `zip -> GEOGRAPHY` (đã nêu trong docx, mục "Đã sửa").
- `payment_method`: dữ liệu thanh toán chính thức chỉ nên nằm ở bảng `PAYMENT` (đã nêu trong docx).
- `sales_employee_name`, `marital_status`, `education_level`, `years_experience`: đây là thuộc tính mô tả nhân viên bán hàng, không thuộc bảng `ORDER`. Lược đồ 3NF hiện tại của `SALES_EMPLOYEE` cũng chỉ có `sales_employee_id` và `name` — các cột nhân khẩu học này **nằm ngoài phạm vi 4 bảng được yêu cầu xử lý** (CUSTOMER, ORDER, ORDER_ITEMS, ORDER_ITEM_PROMOTION), cần bàn thêm với đội thiết kế dữ liệu nếu muốn giữ lại (có thể cần mở rộng bảng SALES_EMPLOYEE).
- `comment`: nội dung nhận xét dạng text tự do, không thuộc lược đồ `ORDER` hiện có (có thể liên quan tới một bảng ghi nhận phản hồi CSKH riêng, chưa được mô hình hóa) — loại khỏi phạm vi xử lý lần này.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW_DIR = Path("./DAAI_N1.4/silver_data_raw")
OUTPUT_DIR = Path("./DAAI_N1.4/silver_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC_FILE = RAW_DIR / "orders_enriched_silver.csv"
CUSTOMER_FILE = OUTPUT_DIR / "CUSTOMER.csv"   # kết quả từ notebook 01
OUT_FILE = OUTPUT_DIR / "ORDER.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 1. Nạp dữ liệu & khảo sát chất lượng (Data Profiling)

In [2]:
df_raw = pd.read_csv(SRC_FILE)
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (646945, 17)


,order_id,order_date,customer_id,zip,city,region,district,order_status,payment_method,device_type,order_source,sales_employee_id,sales_employee_name,marital_status,education_level,years_experience,comment
0,1,2012-07-04,58578,1109,Hanoi,East,District #02,delivered,credit_card,desktop,paid_search,EMP0103,Nguyễn Thị Phúc,Đã kết hôn,Đại học,20,"Dịch vụ chăm sóc khách hàng chuyên nghiệp, thá..."
1,2,2012-07-04,58621,1330,Phu Ly,East,District #02,returned,cod,mobile,paid_search,EMP0180,Lê Gia Hùng,Đã kết hôn,Sau đại học,3,"Dịch vụ ổn định, nhân viên luôn sẵn sàng hỗ tr..."
2,3,2012-07-04,58811,1473,Lao Cai,East,District #02,delivered,credit_card,desktop,direct,EMP0093,Bùi Minh Quân,Đã kết hôn,Cao đẳng,18,Khách hàng không có thêm phản hồi về chất lượn...
3,4,2012-07-04,59453,2360,Son Tay,East,District #02,delivered,credit_card,desktop,referral,EMP0015,Võ Quốc Mai,Độc thân,Trung cấp,12,"Phản hồi thắc mắc đầy đủ, hỗ trợ xuyên suốt qu..."
4,6,2012-07-06,57821,2886,Uong Bi,East,District #02,delivered,paypal,mobile,email_campaign,EMP0107,Hồ Thanh Yến,Độc thân,Đại học,17,Khách hàng đánh giá mức độ hài lòng ở mức khá.


In [3]:
display(df_raw.dtypes)
print()
print("Null theo cột:")
display(df_raw.isnull().sum())

order_id                int64
order_date             object
customer_id             int64
zip                     int64
city                   object
region                 object
district               object
order_status           object
payment_method         object
device_type            object
order_source           object
sales_employee_id      object
sales_employee_name    object
marital_status         object
education_level        object
years_experience        int64
comment                object
dtype: object


Null theo cột:


order_id               0
order_date             0
customer_id            0
zip                    0
city                   0
region                 0
district               0
order_status           0
payment_method         0
device_type            0
order_source           0
sales_employee_id      0
sales_employee_name    0
marital_status         0
education_level        0
years_experience       0
comment                0
dtype: int64

In [4]:
# Kiểm tra khóa chính order_id
n_dup_pk = df_raw['order_id'].duplicated().sum()
n_null_pk = df_raw['order_id'].isnull().sum()
print(f"order_id trùng lặp: {n_dup_pk}")
print(f"order_id null: {n_null_pk}")
assert n_dup_pk == 0 and n_null_pk == 0, "Vi phạm khóa chính order_id!"

order_id trùng lặp: 0
order_id null: 0


In [5]:
# Miền giá trị các cột phân loại giữ lại trong ORDER
for col in ['order_status', 'device_type', 'order_source']:
    print(f"--- {col} ---")
    print(sorted(df_raw[col].unique()))
    print()

# Kiểm tra định dạng sales_employee_id (kỳ vọng dạng EMP + 4 chữ số)
bad_emp = df_raw[~df_raw['sales_employee_id'].astype(str).str.match(r'^EMP\d{4}$')]
print(f"sales_employee_id sai định dạng: {len(bad_emp)}")

invalid_dates = pd.to_datetime(df_raw['order_date'], errors='coerce').isna().sum()
print(f"order_date không parse được: {invalid_dates}")

--- order_status ---
['cancelled', 'created', 'delivered', 'paid', 'returned', 'shipped']

--- device_type ---
['desktop', 'mobile', 'tablet']

--- order_source ---
['direct', 'email_campaign', 'organic_search', 'paid_search', 'referral', 'social_media']

sales_employee_id sai định dạng: 0
order_date không parse được: 0


In [6]:
# Kiểm tra ràng buộc khóa ngoại customer_id với bảng CUSTOMER đã xử lý ở notebook 01
customer_ids = pd.read_csv(CUSTOMER_FILE, usecols=['customer_id'])['customer_id']
orphan_customers = (~df_raw['customer_id'].isin(customer_ids)).sum()
print(f"Số order có customer_id KHÔNG tồn tại trong CUSTOMER: {orphan_customers}")
assert orphan_customers == 0, "Phát hiện customer_id không tồn tại trong bảng CUSTOMER — cần xử lý trước khi nạp!"

Số order có customer_id KHÔNG tồn tại trong CUSTOMER: 0


In [7]:
# Đối chiếu zip của đơn hàng với zip đăng ký của khách hàng (chỉ để quan sát, không phải lỗi)
cust_zip = pd.read_csv(CUSTOMER_FILE, usecols=['customer_id', 'zip']).rename(columns={'zip': 'zip_customer'})
tmp = df_raw.merge(cust_zip, on='customer_id', how='left')
mismatch = (tmp['zip'] != tmp['zip_customer']).sum()
print(f"Số đơn có zip khác với zip đăng ký của khách hàng: {mismatch} / {len(tmp)}")
print("=> Trong dữ liệu hiện tại, zip của đơn hàng luôn trùng với zip của khách hàng đặt đơn.")
print("   Về mặt lược đồ, đây vẫn là 2 khóa ngoại độc lập (đơn có thể giao tới địa chỉ khác nhà),")
print("   chỉ là dữ liệu mẫu hiện chưa phát sinh trường hợp khác nhau.")

Số đơn có zip khác với zip đăng ký của khách hàng: 0 / 646945
=> Trong dữ liệu hiện tại, zip của đơn hàng luôn trùng với zip của khách hàng đặt đơn.
   Về mặt lược đồ, đây vẫn là 2 khóa ngoại độc lập (đơn có thể giao tới địa chỉ khác nhà),
   chỉ là dữ liệu mẫu hiện chưa phát sinh trường hợp khác nhau.


**Nhận xét khảo sát:**
- Không null, `order_id` là khóa duy nhất.
- `order_status`, `device_type`, `order_source` có miền giá trị hợp lệ, không lỗi chính tả.
- `sales_employee_id` đúng 100% định dạng `EMP####`.
- `order_date` parse hợp lệ 100%.
- Toàn bộ `customer_id` trong ORDER đều tồn tại trong CUSTOMER — ràng buộc FK hợp lệ.
- `zip` đơn hàng trùng khớp hoàn toàn với `zip` khách hàng trong dữ liệu hiện tại (ghi nhận để lưu ý, không phải lỗi cần sửa).

## 2. Làm sạch & chuyển đổi theo lược đồ 3NF

In [8]:
df = df_raw.copy()

# (a) Chỉ giữ lại các cột thuộc lược đồ bảng ORDER, loại các cột dư thừa / ngoài phạm vi
cols_to_drop = [
    'city', 'region', 'district',          # dư thừa, suy ra qua zip -> GEOGRAPHY
    'payment_method',                       # thuộc bảng PAYMENT
    'sales_employee_name', 'marital_status', 'education_level', 'years_experience',  # ngoài phạm vi ORDER/SALES_EMPLOYEE hiện tại
    'comment',                              # không thuộc lược đồ ORDER
]
df = df.drop(columns=cols_to_drop)

# (b) Chuẩn hóa chuỗi
str_cols = ['order_status', 'device_type', 'order_source', 'sales_employee_id']
for col in str_cols:
    df[col] = df[col].str.strip()

# (c) Ép kiểu dữ liệu
df['order_id'] = df['order_id'].astype('int64')
df['customer_id'] = df['customer_id'].astype('int64')
df['zip'] = df['zip'].astype('int64')
df['order_date'] = pd.to_datetime(df['order_date']).dt.date

# (d) Loại dòng trùng lặp hoàn toàn (phòng vệ)
before = len(df)
df = df.drop_duplicates()
print(f"Loại bỏ {before - len(df)} dòng trùng lặp hoàn toàn.")

# (e) Sắp xếp cột đúng theo lược đồ ORDER
df = df[['order_id', 'order_date', 'customer_id', 'zip', 'order_status',
         'device_type', 'order_source', 'sales_employee_id']]

df.head()

Loại bỏ 0 dòng trùng lặp hoàn toàn.


,order_id,order_date,customer_id,zip,order_status,device_type,order_source,sales_employee_id
0,1,2012-07-04,58578,1109,delivered,desktop,paid_search,EMP0103
1,2,2012-07-04,58621,1330,returned,mobile,paid_search,EMP0180
2,3,2012-07-04,58811,1473,delivered,desktop,direct,EMP0093
3,4,2012-07-04,59453,2360,delivered,desktop,referral,EMP0015
4,6,2012-07-06,57821,2886,delivered,mobile,email_campaign,EMP0107


## 3. Kiểm tra ràng buộc cuối & xuất bảng Silver chuẩn hóa

In [9]:
assert df['order_id'].is_unique, "order_id không còn là khóa duy nhất sau xử lý!"
for col in df.columns:
    n_null = df[col].isnull().sum()
    assert n_null == 0, f"Cột {col} có {n_null} giá trị null sau xử lý!"

# Kiểm tra lại FK customer_id lần cuối
assert df['customer_id'].isin(customer_ids).all(), "Vẫn còn customer_id mồ côi sau xử lý!"

print("Số dòng cuối cùng:", len(df))
print("Số cột:", len(df.columns))
display(df.dtypes)

Số dòng cuối cùng: 646945
Số cột: 8


order_id              int64
order_date           object
customer_id           int64
zip                   int64
order_status         object
device_type          object
order_source         object
sales_employee_id    object
dtype: object

In [10]:
df.to_csv(OUT_FILE, index=False)
print(f"Đã xuất bảng ORDER (chuẩn Silver, 3NF) tại: {OUT_FILE.resolve()}")
print(f"Số dòng: {len(df):,} | Số cột: {len(df.columns)}")

Đã xuất bảng ORDER (chuẩn Silver, 3NF) tại: D:\TH_DA&AI\DAAI_N1.4\silver_data\ORDER.csv
Số dòng: 646,945 | Số cột: 8
